In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("source_system","")
dbutils.widgets.text("table_name","")
dbutils.widgets.text("landing_timestamp","")
source_system = dbutils.widgets.get("source_system")
table_name = dbutils.widgets.get("table_name")
landing_timestamp = dbutils.widgets.get("landing_timestamp")

In [0]:
dbutils.widgets.text("source_file_format","")
source_file_format = dbutils.widgets.get("source_file_format")

In [0]:
dbutils.fs.ls("/Volumes/adb_caresync/landing/landing/neon_postgres/hospitals/2026-11-20_08:08:05/")

In [0]:
landing_path = f"/Volumes/adb_caresync/landing/landing/{source_system}/{table_name}/{landing_timestamp}/"

try:
    landing_files = dbutils.fs.ls(landing_path)
except Exception as e:
    if "FileNotFound" in str(e) or "does not exist" in str(e).lower():
        landing_files = []
    else:
        raise  # real errors (permissions, etc.) should still fail

if not landing_files:
    msg = f"SKIPPED: no landing files found at {landing_path}"
    print(msg)
    dbutils.notebook.exit(msg)  # ends the notebook successfully, later cells are not run

print("File exists:", landing_path)

In [0]:
file_format = source_file_format.strip().lower()

# If ADF didn't pass a format, infer it from the landed file extensions
if not file_format:
    extensions = {f.name.rsplit(".", 1)[-1].lower() for f in landing_files if "." in f.name}
    if "parquet" in extensions:
        file_format = "parquet"
    elif "csv" in extensions:
        file_format = "csv"
    print("source_file_format not provided, inferred:", file_format)

if file_format == "parquet":
    landing_df = spark.read.format("parquet").load(landing_path)
elif file_format == "csv":
    landing_df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(landing_path)
else:
    raise Exception(f"Invalid file format: {source_file_format}")

print("Data read successfully")

In [0]:
updated_df=landing_df.withColumn("landing_timestamp",lit(landing_timestamp)).withColumn("insert_timestamp",current_timestamp())

In [0]:
updated_df.write.mode("append").saveAsTable(f"adb_caresync.bronze.{table_name}")